# Model checks: the linearized model and the test paths

The model checks from Henry's `ControllerAnalysis_Linearized.ipynb`, on the merged notebooks (the robot test runs are in `ControllerAnalysis.ipynb`):

1. **Loading** (fast: definitions only)
2. **Step 1:** the model's equations f against the plant
3. **Step 2:** the Jacobians $\mathbf{A}$, $\mathbf{B}$
4. **Step 3:** the 20 ms model (20 sub-steps of 1 ms): 0.4 s predictions against the plant, and against Dana's simple model
5. **Path check:** the peak wheel speed each of `ControllerAnalysis.ipynb`'s test paths needs, as a share of the supply's limit


In [ ]:
# Load only the definitions (no test plots): plant, parameters, controllers, path generator, Dana's runner
import json
import time
import numpy as np
import matplotlib.pyplot as plt


def notebook_cells(path):
    """The source of every cell of a notebook, in order."""
    with open(path, encoding="utf-8") as f:
        cells = json.load(f)["cells"]
    sources = ["".join(c["source"]) for c in cells]
    return sources


def notebook_cells_by_id(path):
    """The source of every cell of a notebook, by cell id."""
    with open(path, encoding="utf-8") as f:
        cells = json.load(f)["cells"]
    sources = {c.get("id"): "".join(c["source"]) for c in cells}
    return sources


# Plant: the observable state, both plant classes, the parameters, model_choice
plant_cells = notebook_cells_by_id("PlantModels.ipynb")
for cell_id in ("b5584b9d", "e4823acb", "ce831979", "f56abec6", "bf71ff62"):
    exec(plant_cells[cell_id])

# Controllers: every code cell of the models notebook (Dana's PID and MPC, the linearized model and MPC)
for source in notebook_cells("ControllerModels.ipynb"):
    if source.lstrip().startswith(("import", "class")) or "\nclass " in source:     # also cells with a comment first (RiccatiTerminalCost)
        exec(source)

# Path generator (Dana's class only, not her demo plots)
for source in notebook_cells("PathGeneration.ipynb"):
    if "class PathGenerator" in source:
        exec(source)


print("Loaded: plant, parameters, controller classes, path generator")


## Step 1 check: does $f$ agree with the plant?

In [ ]:
# Step 1 check: f at the plant's new state against the plant's own 1 ms step
# defined in the notebooks loaded with %run above; declared global so the editor sees them (as Dana does)
global simulation_parameters, OmniRobotSimulationODESolve, OmniRobotStateODESolve, OmniRobotLinearizedModel
check_params = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0, implicit_tol=1e-12, implicit_max_iter=200)
model = OmniRobotLinearizedModel(check_params)
plant = OmniRobotSimulationODESolve(check_params)
state = OmniRobotStateODESolve()
dt = check_params['dt']
V_strafe = np.array([-6.0, -6.0, 6.0, 6.0])

names = ["pose", "velocity", "currents", "wheel speeds", "bristles"]
parts = [model.POSE, model.VELOCITY, model.CURRENTS, model.WHEELS, model.BRISTLES]
worst_gap = np.zeros(5)
worst_matched = np.zeros(5)
largest_rate = np.zeros(5)
sliding_steps = 0
worst_load_gap = 0.0

for k in range(1000):
    V = V_strafe if k < 500 else -V_strafe            # reverse at 0.5 s so the wheels slide
    F_n = plant.compute_dynamic_normal_forces(state)  # the loads the plant uses this step (no noise)
    x_now = model.get_state_from_plant(state)
    state = plant.step(state, V)
    x_next = model.get_state_from_plant(state)
    plant_rate = (x_next - x_now) / dt

    # (a) f at the plant's new state, as it is
    bristle_decay = model.get_bristle_decay(x_next)
    model_rate = model.get_state_derivative(x_next, V, bristle_decay, F_n=F_n)

    # (b) f given the start-of-step values the plant uses in those terms
    matched_rate = model_rate.copy()
    #   velocity row: the forces rotated with the OLD heading (plant: R_theta at the step start), and the roller
    #   friction from the OLD velocity (plant: compute_roller_friction uses state.v_local at the step start)
    x_old_heading = x_next.copy()
    x_old_heading[2] = x_now[2]
    x_old_heading[model.VELOCITY] = x_now[model.VELOCITY]
    matched_rate[model.VELOCITY] = model.get_state_derivative(x_old_heading, V, bristle_decay, F_n=F_n)[model.VELOCITY]
    #   bristle row: the slip with the OLD velocity and heading (plant: v_center_d = H_drive v_local at the step start)
    x_old_velocity = x_next.copy()
    x_old_velocity[2] = x_now[2]
    x_old_velocity[model.VELOCITY] = x_now[model.VELOCITY]
    bristle_decay_old_v = model.get_bristle_decay(x_old_velocity)
    matched_rate[model.BRISTLES] = model.get_state_derivative(x_old_velocity, V, bristle_decay_old_v, F_n=F_n)[model.BRISTLES]
    #   wheel row: the rolling drag with the OLD wheel speed (plant: T_roll from state.omega at the step start)
    T_roll_old = check_params['Crr'] * F_n * check_params['r'] * np.tanh(x_now[model.WHEELS] / check_params['roll_smoothing'])
    F_floor_next = -check_params['mu_s'] * F_n * x_next[model.BRISTLES]
    matched_rate[model.WHEELS] = (check_params['Kt'] * x_next[model.CURRENTS] - T_roll_old - check_params['r'] * F_floor_next) / check_params['Jw']

    for n, part in enumerate(parts):
        worst_gap[n] = max(worst_gap[n], np.max(np.abs(model_rate[part] - plant_rate[part])))
        worst_matched[n] = max(worst_matched[n], np.max(np.abs(matched_rate[part] - plant_rate[part])))
        largest_rate[n] = max(largest_rate[n], np.max(np.abs(plant_rate[part])))
    sliding_steps += np.any(state.S)
    worst_load_gap = max(worst_load_gap, np.max(np.abs(model.get_wheel_loads(x_now) - F_n)))    # model's loads vs the plant's, same step

print(f"1000 plant steps, {sliding_steps} with a wheel sliding")
print(f"{'state':14s} {'largest rate':>13s} | {'(a) gap':>10s} {'/ rate':>9s} | {'(b) gap':>10s} {'/ rate':>9s}")
for n in range(5):
    print(f"{names[n]:14s} {largest_rate[n]:13.4g} | {worst_gap[n]:10.3g} {worst_gap[n] / largest_rate[n]:9.1e} | "
          f"{worst_matched[n]:10.3g} {worst_matched[n] / largest_rate[n]:9.1e}")
print(f"wheel loads: the model's (from its lagged acceleration state) vs the plant's, same step: largest gap {worst_load_gap:.3g} N")


In [ ]:
# Step 1 plot: one rate from each part of the state, plant against model (with the plant's start-of-step values)
plot_params = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0, implicit_tol=1e-12, implicit_max_iter=200)
model = OmniRobotLinearizedModel(plot_params)
plant = OmniRobotSimulationODESolve(plot_params)
state = OmniRobotStateODESolve()
dt = plot_params['dt']
V_strafe = np.array([-6.0, -6.0, 6.0, 6.0])
rows = [("x position rate (m/s)", 0), ("x velocity rate (m/s^2)", 3), ("wheel 1 current rate (A/s)", 6),
        ("wheel 1 speed rate (rad/s^2)", 10), ("wheel 1 bristle rate (1/s)", 14)]
plant_series, model_series, sliding = [], [], []
for k in range(1000):
    V = V_strafe if k < 500 else -V_strafe
    F_n = plant.compute_dynamic_normal_forces(state)
    x_now = model.get_state_from_plant(state)
    state = plant.step(state, V)
    x_next = model.get_state_from_plant(state)
    plant_rate = (x_next - x_now) / dt
    # the model's rate, given the plant's start-of-step values in the terms where the plant uses them
    x_matched = x_next.copy()
    x_matched[2] = x_now[2]
    x_matched[model.VELOCITY] = x_now[model.VELOCITY]
    d_matched = model.get_bristle_decay(x_matched)
    rate = model.get_state_derivative(x_next, V, d_matched, F_n=F_n)
    rate[model.VELOCITY] = model.get_state_derivative(x_matched, V, d_matched, F_n=F_n)[model.VELOCITY]
    rate[model.BRISTLES] = model.get_state_derivative(x_matched, V, d_matched, F_n=F_n)[model.BRISTLES]
    T_roll_old = plot_params['Crr'] * F_n * plot_params['r'] * np.tanh(x_now[model.WHEELS] / plot_params['roll_smoothing'])
    rate[model.WHEELS] = (plot_params['Kt'] * x_next[model.CURRENTS] - T_roll_old + plot_params['r'] * plot_params['mu_s'] * F_n * x_next[model.BRISTLES]) / plot_params['Jw']
    plant_series.append([plant_rate[i] for _, i in rows])
    model_series.append([rate[i] for _, i in rows])
    sliding.append(np.any(state.S))
plant_series, model_series, sliding = np.array(plant_series), np.array(model_series), np.array(sliding)
t = dt * np.arange(1, 1001)

fig, axs = plt.subplots(len(rows), 1, figsize=(12, 14), sharex=True)
for n, (label, _) in enumerate(rows):
    axs[n].fill_between(t, 0, 1, where=sliding, color='orange', alpha=0.12, transform=axs[n].get_xaxis_transform(), label='a wheel sliding')
    axs[n].plot(t, plant_series[:, n], color='black', lw=2.5, alpha=0.5, label='plant')
    axs[n].plot(t, model_series[:, n], color='seagreen', lw=1.0, ls='--', label='model f')
    axs[n].set_ylabel(label); axs[n].grid(True, linestyle=':', alpha=0.6)
axs[0].legend(loc='upper right'); axs[-1].set_xlabel("Time (s): 6 V strafe, reversed at 0.5 s")
axs[0].set_title("Step 1: the model's rates on top of the plant's")
plt.tight_layout()
plt.show()


## Step 2 check: the Jacobians

In [ ]:
# Step 2 check: the Jacobians against entries worked out by hand, and against small changes in f
# defined in the notebooks loaded with %run above; declared global so the editor sees them (as Dana does)
global simulation_parameters, OmniRobotSimulationODESolve, OmniRobotStateODESolve, OmniRobotLinearizedModel
check_params = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0)
model = OmniRobotLinearizedModel(check_params)
plant = OmniRobotSimulationODESolve(check_params)
state = OmniRobotStateODESolve()
V_strafe = np.array([-6.0, -6.0, 6.0, 6.0])

# Run the plant and keep two points: gripping (0.30 s, before the reversal) and sliding (0.55 s, just after it)
points = {}
for k in range(600):
    V = V_strafe if k < 500 else -V_strafe
    state = plant.step(state, V)
    if k == 299:
        points['gripping'] = (model.get_state_from_plant(state), V.copy(), state.a_local_past.copy(), state.S.copy())
    if k == 549:
        points['sliding'] = (model.get_state_from_plant(state), V.copy(), state.a_local_past.copy(), state.S.copy())

p = check_params
rng = np.random.default_rng(0)
for name, (x, u, a_local_past, sliding_flags) in points.items():
    bristle_decay = model.get_bristle_decay(x)
    F_n = model.get_wheel_loads(x)
    A, B = model.get_continuous_matrices(x, u, bristle_decay, F_n=F_n)          # (a): loads held, so the hand formulas are exact
    A_full, B_full = model.get_continuous_matrices(x, u, bristle_decay)         # (b): the model as the MPC uses it, loads differentiated

    # (a) entries worked out by hand, wheel 1 (index 0)
    hand = [
        ("d(pose rate)/d(velocity), x", A[0, 3], 1.0),
        ("d(I_dot)/dV = 1/Lm", B[6, 0], 1.0 / p['Lm']),
        ("d(I_dot)/dI = -R/Lm", A[6, 6], -p['R'] / p['Lm']),
        ("d(I_dot)/d(omega) = -Kb/Lm", A[6, 10], -p['Kb'] / p['Lm']),
        ("d(omega_dot)/dI = Kt/Jw", A[10, 6], p['Kt'] / p['Jw']),
        ("d(omega_dot)/dz = r mu_s F_n/Jw", A[10, 14], p['r'] * p['mu_s'] * F_n[0] / p['Jw']),
        ("d(z_dot)/d(omega) = -r/delta", A[14, 10], -p['r'] / p['lugre_deflection']),
        ("d(z_dot)/dz = -d/delta", A[14, 14], -bristle_decay[0] / p['lugre_deflection']),
    ]
    print(f"\n{name}: wheels sliding {sliding_flags}")
    print(f"   {'entry':34s} {'A_c, B_c':>14s} {'by hand':>14s} {'difference':>11s}")
    for label, numeric, by_hand in hand:
        print(f"   {label:34s} {numeric:14.6g} {by_hand:14.6g} {abs(numeric - by_hand) / abs(by_hand):11.1e}")

    # (b) small changes: scaled to each part of the state
    scale = np.concatenate((1e-4 * np.ones(6), 1e-4 * np.ones(4), 1e-3 * np.ones(4), 1e-4 * np.ones(4), 1e-3 * np.ones(2)))
    worst = 0.0
    for trial in range(20):
        dx = scale * rng.standard_normal(model.n_x)
        du = 1e-3 * rng.standard_normal(4)
        real_change = model.get_state_derivative(x + dx, u + du, bristle_decay) - model.get_state_derivative(x, u, bristle_decay)
        line_change = A_full @ dx + B_full @ du
        worst = max(worst, np.linalg.norm(line_change - real_change) / np.linalg.norm(real_change))
    print(f"   (b) small changes, 20 random: largest mismatch {worst:.1e} of the change")


In [ ]:
# Step 2 plot: the A matrix while gripping and while sliding (log10 of |entry|; blank = zero)
state_labels = ['X', 'Y', 'th', 'Xd', 'Yd', 'thd', 'I1', 'I2', 'I3', 'I4', 'w1', 'w2', 'w3', 'w4', 'z1', 'z2', 'z3', 'z4', 'a_r', 'a_f']
fig, axs = plt.subplots(1, 2, figsize=(16, 7))
for ax, (name, (x, u, a_local_past, sliding_flags)) in zip(axs, points.items()):
    bristle_decay = model.get_bristle_decay(x)
    A, B = model.get_continuous_matrices(x, u, bristle_decay)        # loads differentiated, as the MPC uses it
    magnitude = np.where(np.abs(A) > 1e-9, np.log10(np.abs(A) + 1e-300), np.nan)
    image = ax.imshow(magnitude, cmap='viridis', vmin=-3, vmax=6)
    ax.set_xticks(range(len(state_labels))); ax.set_xticklabels(state_labels, rotation=90)
    ax.set_yticks(range(len(state_labels))); ax.set_yticklabels([lbl + "'" for lbl in state_labels])
    ax.set_title(f"A while {name} (sliding {sliding_flags.astype(int)})")
    ax.set_xlabel("how much each state ...") ; ax.set_ylabel("... changes each rate")
fig.colorbar(image, ax=axs, label="log10 |A entry|", shrink=0.8)
plt.show()


## Step 3 check: the 20 ms model, 0.4 s ahead

From the plant's own states (noise off), each model predicts 0.4 s ahead (20 controller steps) with the voltages the plant actually got, and the predicted position is compared with where the plant went.

* **Linearized:** `get_discrete_matrices` 20 times, as the MPC predicts: each step 20 backward-Euler sub-steps of 1 ms, the Jacobians every 5 ms, the relax rate $d$ every 1 ms.
* **Dana's simple model:** her `OmniRobotNominalModel` (global velocity), 20 steps of $(\mathbf{A}_d, \mathbf{B}_d)$ at the predicted heading.

Three manoeuvres: a 10 V strafe (the wheels slide), a 4 V side-to-side reversal, and random 8 V steps every 0.2 s.


In [ ]:
# Step 3 check: 0.4 s predictions against the plant, linearized model and Dana's simple model
pred_params = dict(simulation_parameters, sigma_V=0.0, sigma_mu=0.0)
lin_model = OmniRobotLinearizedModel(pred_params)
simple_model = OmniRobotNominalModel(pred_params)
Ts, horizon = 0.02, 20
substeps_per_step = int(round(Ts / pred_params['dt']))
STRAFE = np.array([-1.0, -1.0, 1.0, 1.0])
random_voltages = np.random.default_rng(3).uniform(-8, 8, (20, 4))
manoeuvres = {
    "10 V strafe": (lambda t: 10 * STRAFE, 2.0),
    "4 V side-to-side": (lambda t: -4 * STRAFE * (1.0 if t < 1 else (-1.0 if t < 2 else 0.0)), 3.0),
    "random 8 V steps": (lambda t: random_voltages[min(int(t / 0.2), 19)], 4.0),
}

prediction_results = {}
for name, (V_of_t, duration) in manoeuvres.items():
    plant = OmniRobotSimulationODESolve(pred_params)
    state = OmniRobotStateODESolve()
    states, voltages = [], []
    for k in range(int(round(duration / Ts))):
        V = V_of_t(k * Ts)
        states.append(state)
        voltages.append(V)
        for _ in range(substeps_per_step):
            state = plant.step(state, V)
    states.append(state)

    lin_err, simple_err, starts = [], [], []
    for k in range(5, len(voltages) - horizon, 5):
        # linearized model
        x = lin_model.get_state_from_plant(states[k])
        for j in range(horizon):
            _, _, _, x = lin_model.get_discrete_matrices(x, voltages[k + j], Ts, 20)   # as the MPC: 1 ms sub-steps, Jacobians every 5 ms
        lin_err.append(np.hypot(*(x[:2] - states[k + horizon].pos_global[:2])))
        # Dana's simple model (global velocity)
        xs = np.concatenate((states[k].pos_global, states[k].v_global, states[k].I))
        for j in range(horizon):
            A_d, B_d = simple_model.get_discrete_matrices(xs[2], Ts)
            xs = A_d @ xs + B_d @ voltages[k + j]
        simple_err.append(np.hypot(*(xs[:2] - states[k + horizon].pos_global[:2])))
        starts.append(k * Ts)
    prediction_results[name] = (np.array(starts), 1e3 * np.array(lin_err), 1e3 * np.array(simple_err))
    print(f"{name:18s} position error after 0.4 s: linearized median {np.median(lin_err) * 1e3:6.1f} mm (worst {np.max(lin_err) * 1e3:6.1f}) | "
          f"Dana's simple model median {np.median(simple_err) * 1e3:6.1f} mm (worst {np.max(simple_err) * 1e3:6.1f})")

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, (starts, lin_err, simple_err)) in zip(axs, prediction_results.items()):
    ax.plot(starts, simple_err, 'o-', color='darkorange', label="Dana's simple model")
    ax.plot(starts, lin_err, 'o-', color='seagreen', label='linearized model')
    ax.set_yscale('log'); ax.grid(True, which='both', linestyle=':', alpha=0.6)
    ax.set_title(name); ax.set_xlabel("prediction start time (s)"); ax.set_ylabel("position error after 0.4 s (mm)")
    ax.legend()
plt.tight_layout()
plt.show()


## Path check: the wheel speed each test path needs

The test paths of `ControllerAnalysis.ipynb` (made the same way here), and the peak wheel speed each one needs, as a share of the supply's limit $V_{max} / K_b$ (where the back-EMF takes the whole supply): $\omega = \mathbf{H}_{drive}\,\mathbf{R}(\theta)^\top \dot{\mathbf{X}}_g / r$ at each step. Above 100% the robot cannot keep up with the path at full voltage.


In [ ]:
# Path check: the peak wheel speed each of ControllerAnalysis.ipynb's test paths needs (the same paths, made here)
HOLD_DURATION = 0.5
path_generator = PathGeneratorWithHold(dt=0.02)


def figure_eight_path(size, period, laps=2, ramp=0.3, swing=0.3, dt=0.02):
    """Figure eight in Dana's path format (t, pos, vel; global frame): size [m], period per lap [s] (as in ControllerAnalysis.ipynb)."""
    t = np.arange(0.0, laps * period + 1.0, dt)
    rate = 2 * np.pi / period
    phase = rate * (t - ramp * (1 - np.exp(-t / ramp)))
    phase_dot = rate * (1 - np.exp(-t / ramp))
    pos = np.column_stack((size * np.sin(phase), 0.5 * size * np.sin(2 * phase), swing * np.sin(phase)))
    vel = np.column_stack((size * np.cos(phase) * phase_dot, size * np.cos(2 * phase) * phase_dot, swing * np.cos(phase) * phase_dot))
    path = {'t': t, 'pos': pos, 'vel': vel}
    return path


def peak_wheel_speed_share(path):
    """
    The peak wheel speed a path needs, as a share of the supply's limit V_max / Kb: at each step the velocity along the
    robot's axes, v_local = R(theta)^T v_global, then the wheel speeds omega = H_drive v_local / r.
    """
    geometry = OmniRobotSimulationODESolve(simulation_parameters)
    omega_needed = np.zeros((len(path['t']), 4))
    for k in range(len(path['t'])):
        theta = path['pos'][k, 2]
        rotation_matrix = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                                    [np.sin(theta),  np.cos(theta), 0.0],
                                    [0.0,            0.0,           1.0]])
        v_local = rotation_matrix.T @ path['vel'][k]
        omega_needed[k] = geometry.H_drive @ v_local / simulation_parameters['r']
    omega_limit = simulation_parameters['V_max'] / simulation_parameters['Kb']
    share = np.abs(omega_needed).max() / omega_limit
    return share


arena_half = 9.0 / 2.0
arena_margin = 0.5
workspace_9m = {'x_min': -arena_half + arena_margin, 'x_max': arena_half - arena_margin,
                'y_min': -arena_half + arena_margin, 'y_max': arena_half - arena_margin, 'theta_min': -np.pi, 'theta_max': np.pi}
quadrant_waypoints = [np.array([0.0, 0.0, 0.0]), np.array([3.2, 2.8, np.radians(90.0)]), np.array([-2.5, 3.5, np.radians(180.0)]),
                      np.array([-3.5, -2.8, np.radians(-90.0)]), np.array([2.8, -3.2, 0.0])]
small_area = {'x_min': -0.5, 'x_max': 0.5, 'y_min': -0.5, 'y_max': 0.5, 'theta_min': -np.pi / 2, 'theta_max': np.pi / 2}
dana_waypoints = [np.array([0.0, 0.0, 0.0]), np.array([2.2, 1.2, 0.50]), np.array([1.0, 3.5, -0.60]), np.array([3.2, 2.8, 0.785])]

check_paths = {
    "Dana's waypoints, right-angle corners (5 s + 0.5 s hold)": PathGeneratorCurvedWithHold(dt=0.02).generate_waypoints_sequence(dana_waypoints, segment_duration=5.0, segment_hold=HOLD_DURATION),
    "Dana 8 s": path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, np.pi / 4])], segment_duration=8.0, segment_hold=HOLD_DURATION),
    "Dana 4 s (too fast)": path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, np.pi / 4])], segment_duration=4.0, segment_hold=HOLD_DURATION),
    "Random (10 s segments)": path_generator.generate_random_sequence(num_segments=4, workspace_bounds=workspace_9m, segment_duration=10.0, segment_hold=HOLD_DURATION, random_seed=42),
    "Quadrants (12 s segments)": path_generator.generate_waypoints_sequence(quadrant_waypoints, segment_duration=12.0, segment_hold=HOLD_DURATION),
    "Quick hops (1.25 s)": path_generator.generate_random_sequence(num_segments=6, workspace_bounds=small_area, segment_duration=1.25, segment_hold=HOLD_DURATION, random_seed=7),
    "Figure eight 0.4 m (3.4 s)": figure_eight_path(0.4, 3.4),
    "Figure eight 0.4 m (2.5 s)": figure_eight_path(0.4, 2.5),
    "Figure eight 0.3 m (2.2 s)": figure_eight_path(0.3, 2.2),
}
for theta_end, duration in [(np.pi / 2, 8.0), (np.pi / 2, 5.0), (np.pi / 2, 4.0), (np.pi, 8.0), (np.pi, 5.0), (np.pi, 4.0)]:   # the turning test's cases
    name = f"to (3, 3, {'pi/2' if theta_end < 2 else 'pi'}) in {duration:.0f} s"
    check_paths[name] = path_generator.generate_waypoints_sequence([np.zeros(3), np.array([3.0, 3.0, theta_end])],
                                                                   segment_duration=duration, segment_hold=HOLD_DURATION)

print(f"{'Path':38s} {'Peak wheel speed (of the supply limit)':>40s}")
for name, path in check_paths.items():
    share = peak_wheel_speed_share(path)
    print(f"{name:38s} {100 * share:39.0f}%")
